In [1]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer
import string
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [2]:
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()

    # Tokenize text into words
    words = word_tokenize(text)

    # Remove punctuation and stopwords
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words and word not in string.punctuation]

    # Apply stemming
    ps = PorterStemmer()
    words = [ps.stem(word) for word in words]

    return ' '.join(words)


In [3]:
def rank_sentences(text, tfidf_matrix, feature_names):
    sentences = sent_tokenize(text)

    # Preprocess sentences for TF-IDF vectorizer compatibility
    preprocessed_sentences = [preprocess_text(sentence) for sentence in sentences]

    # Transform sentences into TF-IDF vectors
    vectorizer = TfidfVectorizer(vocabulary=feature_names)
    sentence_vectors = vectorizer.fit_transform(preprocessed_sentences)

    # Sum TF-IDF scores for each sentence to get sentence importance
    sentence_scores = sentence_vectors.sum(axis=1).A1

    # Pair sentences with their scores
    ranked_sentences = [(sentences[i], sentence_scores[i]) for i in range(len(sentences))]

    # Sort sentences by score (descending)
    ranked_sentences.sort(key=lambda x: x[1], reverse=True)

    return ranked_sentences

In [4]:
def generate_summary(text, top_n=3):
    # Preprocess entire text (document)
    processed_text = preprocess_text(text)

    # Compute TF-IDF for the whole document (only one document here)
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([processed_text])

    # Get feature names (words)
    feature_names = vectorizer.get_feature_names_out()
    # print(feature_names)
    # Rank sentences based on TF-IDF scores
    ranked_sentences = rank_sentences(text, tfidf_matrix, feature_names)

    # Select top_n sentences for the summary
    summary_sentences = ["   "+sent+"\n" for sent, score in ranked_sentences[:top_n]]

    # Join to form the summary
    summary = ' '.join(summary_sentences)

    return summary

In [ ]:
text="""The debate over universal basic income (UBI) has gained momentum amid the economic uncertainties brought on by the COVID-19 pandemic. Advocates argue that UBI, which involves providing all citizens with a regular, unconditional sum of money, can alleviate poverty and reduce inequality. "Universal basic income is not just a policy; it's a social justice movement," stated Andrew Yang, a former U.S. presidential candidate and prominent UBI supporter. Countries like Finland and Spain have conducted pilot programs to test the feasibility and impact of UBI, with mixed results. Supporters highlight the potential benefits, such as improved mental health, increased financial security, and greater economic stability.
Critics, however, caution against the high costs and potential disincentives to work. They argue that UBI could lead to inflation and reduce the motivation for people to seek employment. Additionally, there are concerns about how to fund such a program sustainably. Economists suggest various funding mechanisms, including higher taxes on wealth and income, as well as the reallocation of existing welfare budgets. As policymakers and researchers continue to explore the implications of UBI, the debate remains a contentious and highly relevant topic in discussions about the future of social welfare and economic policy."""

In [5]:
text="""Ad sales boost Time Warner profit.

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.

The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.

Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service free to TimeWarner internet customers and will try to sign up AOL's existing customers for high-speed broadband. TimeWarner also has to restate 2000 and 2003 results following a probe by the US Securities Exchange Commission (SEC), which is close to concluding.

Time Warner's fourth quarter profits were slightly better than analysts' expectations. But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and Catwoman, a sharp contrast to year-earlier, when the third and final film in the Lord of the Rings trilogy boosted results. For the full-year, TimeWarner posted a profit of $3.36bn, up 27% from its 2003 performance, while revenues grew 6.4% to $42.09bn. "Our financial performance was strong, meeting or exceeding all of our full-year objectives and greatly enhancing our flexibility," chairman and chief executive Richard Parsons said. For 2005, TimeWarner is projecting operating earnings growth of around 5%, and also expects higher revenue and wider profit margins.

TimeWarner is to restate its accounts as part of efforts to resolve an inquiry into AOL by US market regulators. It has already offered to pay $300m to settle charges, in a deal that is under review by the SEC. The company said it was unable to estimate the amount it needed to set aside for legal reserves, which it previously set at $500m. It intends to adjust the way it accounts for a deal with German music publisher Bertelsmann's purchase of a stake in AOL Europe, which it had reported as advertising revenue. It will now book the sale of its stake in AOL Europe as a loss on the value of that stake."""

In [6]:
# from google.colab import files

# uploaded = files.upload()

# for filename in uploaded.keys():
#     with open(filename, 'r', encoding='utf-8') as file:
        # document_text = file.read()
summary = generate_summary(text, top_n=3)
print(f"Generated Summary :", summary)

Generated Summary :    But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and Catwoman, a sharp contrast to year-earlier, when the third and final film in the Lord of the Rings trilogy boosted results.
    "Our financial performance was strong, meeting or exceeding all of our full-year objectives and greatly enhancing our flexibility," chairman and chief executive Richard Parsons said.
    It intends to adjust the way it accounts for a deal with German music publisher Bertelsmann's purchase of a stake in AOL Europe, which it had reported as advertising revenue.



In [ ]:
# @title
from sklearn.metrics.pairwise import cosine_similarity

def generate_summary_cosine(text, top_n=3):
    sentences = sent_tokenize(text)

    # Preprocess sentences
    preprocessed_sentences = [preprocess_text(sentence) for sentence in sentences]

    # Compute TF-IDF for preprocessed sentences
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(preprocessed_sentences)

    # Compute cosine similarity between sentences
    cosine_sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

    # Calculate sentence scores by summing cosine similarities for each sentence
    sentence_scores = cosine_sim_matrix.sum(axis=1)

    # Pair sentences with their scores
    ranked_sentences = [(sentences[i], sentence_scores[i]) for i in range(len(sentences))]

    # Sort sentences by score (descending)
    ranked_sentences.sort(key=lambda x: x[1], reverse=True)

    # Select top_n sentences for the summary
    summary_sentences = ["   "+sent+"\n" for sent, score in ranked_sentences[:top_n]]

    # Join to form the summary
    summary = ' '.join(summary_sentences)

    return summary

In [ ]:
# @title
# from google.colab import files

# uploaded = files.upload()

# for filename in uploaded.keys():
#     with open(filename, 'r', encoding='utf-8') as file:
#         document_text = file.read()
#         summary = generate_summary_cosine(text, top_n=3)
#         print(f"Generated Summary for {filename}:\n\n", summary)
summary = generate_summary_cosine(text, top_n=3)
print(f"Generated Summary for provided text:\n\n", summary)

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import nltk

# Your original text
# text = """
# The ocean is home to a diverse range of species, from the smallest plankton
# to the enormous blue whale. These marine ecosystems are vital for regulating
# the global climate. Unfortunately, plastic pollution poses a significant
# threat to marine life. Many animals, such as sea turtles and birds, mistake
# plastic debris for food, which can be fatal. Urgent action is needed to
# reduce plastic waste and protect our oceans.
# """

# 1. Split text into sentences
sentences = nltk.sent_tokenize(text)

# 2. Load a BERT-based model to get embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
sentence_embeddings = model.encode(sentences)

# 3. Build the Cosine Similarity Matrix
sim_matrix = cosine_similarity(sentence_embeddings)

# 4. Calculate sentence scores by summing their similarity to all others
# (We use sim_matrix.shape[0] as the number of sentences)
sentence_scores = np.zeros(sim_matrix.shape[0])

for i in range(sim_matrix.shape[0]):
    for j in range(sim_matrix.shape[0]):
        if i != j: # Don't compare a sentence to itself
            sentence_scores[i] += sim_matrix[i][j]

# 5. Rank the sentences and get the top N
num_sentences = 3
# Use argsort to get the indices of the top N scores, in descending order
top_sentence_indices = np.argsort(sentence_scores)[::-1][:num_sentences]

# Sort the indices so the summary appears in the original text's order
top_sentence_indices = sorted(top_sentence_indices)

# 6. Select and print the summary
summary = " ".join(["   "+sentences[i]+"\n" for i in top_sentence_indices])

print("--- Extractive Summary ---")
print(summary)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- Extractive Summary ---
   Ad sales boost Time Warner profit.
    Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.
    For the full-year, TimeWarner posted a profit of $3.36bn, up 27% from its 2003 performance, while revenues grew 6.4% to $42.09bn.



In [ ]:
expected_summary="TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn.For the full-year, TimeWarner posted a profit of $3.36bn, up 27% from its 2003 performance, while revenues grew 6.4% to $42.09bn.Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues.Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.For 2005, TimeWarner is projecting operating earnings growth of around 5%, and also expects higher revenue and wider profit margins.It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. Time Warner's fourth quarter profits were slightly better than analysts' expectations."